# AKIS-v2 — Colab Execution Guide

**Runtime:** CPU only  
**Estimated setup time:** ~5 min (first run downloads models)

## Quickstart
1. Upload your PDF when prompted
2. Run all cells in order
3. Use the query cell at the bottom to test

In [ ]:
# ── CELL 1: Clone or upload project ──────────────────────────────────
# Option A: clone from GitHub (replace with your repo URL)
# !git clone https://github.com/YOUR_USERNAME/AKIS_v2.git
# %cd AKIS_v2

# Option B: upload the zip, then extract
from google.colab import files
print('Upload AKIS_v2.zip or your project folder zip')
uploaded = files.upload()
import zipfile, os
for fname in uploaded:
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('.')
print('Extracted:', os.listdir('.'))

In [ ]:
# ── CELL 2: Install dependencies ────────────────────────────────────
!pip install -q \
  fastapi==0.110.0 \
  uvicorn==0.29.0 \
  faiss-cpu==1.7.4 \
  sentence-transformers==2.7.0 \
  rank-bm25==0.2.2 \
  pdfplumber==0.10.3 \
  PyYAML==6.0.1 \
  pydantic==2.6.4 \
  python-multipart==0.0.9 \
  requests==2.31.0 \
  streamlit==1.33.0
print('Installation complete')

In [ ]:
# ── CELL 3: Set PYTHONPATH ───────────────────────────────────────────
import sys, os
# Adjust if your folder name differs
sys.path.insert(0, os.path.abspath('AKIS_v2'))
os.chdir('AKIS_v2')
print('Working dir:', os.getcwd())

In [ ]:
# ── CELL 4: Upload your PDF ──────────────────────────────────────────
from google.colab import files
print('Upload your PDF file:')
pdf_uploaded = files.upload()
pdf_path = list(pdf_uploaded.keys())[0]
print(f'PDF: {pdf_path}')

In [ ]:
# ── CELL 5: Ingest PDF and build pipeline ────────────────────────────
from ingestion.loader import load_pdf
from ingestion.chunker import chunk_text
from orchestrator.pipeline import Pipeline

print('Loading PDF...')
text = load_pdf(pdf_path)
print(f'Extracted {len(text)} characters')

print('Chunking...')
chunks = chunk_text(text, source_file=pdf_path)
print(f'Created {len(chunks)} chunks')

print('Building pipeline (downloads embedding models on first run)...')
pipeline = Pipeline()
pipeline.index(chunks)
print('Pipeline ready!')

In [ ]:
# ── CELL 6: Run a query ──────────────────────────────────────────────
# NOTE: Without Ollama/Gemini configured, generation returns None.
# The pipeline still exercises retrieval, reranking, and evaluation.
# Set OLLAMA_MODEL env var or GEMINI_API_KEY to enable generation.

import os
# Uncomment and set your Gemini key for free LLM generation:
# os.environ['GEMINI_API_KEY'] = 'YOUR_KEY_HERE'

query = 'What are the main skills mentioned in the document?'
result = pipeline.run(query)

print(f'Trace ID   : {result.trace_id}')
print(f'Status     : {result.status}')
print(f'Confidence : {result.confidence}%')
print(f'Retries    : {result.retries}')
print(f'Answer     : {result.answer[:400]}')
print(f'Explanation: {result.explanation}')
print(f'Sources    : {len(result.sources)} chunks retrieved')

In [ ]:
# ── CELL 7: Inspect claim-level grounding ────────────────────────────
print('\nClaim Analysis:')
for c in result.claims:
    icon = '✅' if c['supported'] else '❌'
    print(f"  {icon} [{c['support_level']}] conf={c['confidence']:.2f} | {c['claim'][:80]}")

In [ ]:
# ── CELL 8: Start FastAPI (background thread) ────────────────────────
import threading, uvicorn, os
os.environ['AKIS_DEFAULT_PDF'] = pdf_path

def run_server():
    uvicorn.run('api.main:app', host='0.0.0.0', port=8000, reload=False)

t = threading.Thread(target=run_server, daemon=True)
t.start()
import time; time.sleep(3)
print('FastAPI running at http://localhost:8000')

In [ ]:
# ── CELL 9: Test API endpoint ────────────────────────────────────────
import requests
resp = requests.post('http://localhost:8000/query',
                     json={'query': 'What is this document about?'})
print(resp.json())

In [ ]:
# ── CELL 10: Run benchmark evaluation ───────────────────────────────
from eval.benchmark import run_benchmark
report = run_benchmark('http://localhost:8000')
print('Summary:', report['summary'])